# Install dependencies from uv and setup reloading of imports

In [24]:
!uv sync
%load_ext autoreload
%autoreload 2

Resolved 133 packages in 4ms
Checked 130 packages in 18ms
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Create the agents

In [25]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search, agent_tool

import config
import callbacks
import instructions

research_agent_tool = Agent(name="research_agent_tool",
                            model=config.GEMINI_MODEL,
                            instruction=instructions.get_agent_instructions("research-agent"),
                            before_model_callback=callbacks.log_agent_name_before_callback,
                            tools=[google_search])

research_agent = Agent(
    name="research_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("research-agent"),
    before_model_callback=callbacks.log_agent_name_before_callback,
    tools=[agent_tool.AgentTool(agent=research_agent_tool)]
)

critic_agent = Agent(
    name="critic_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("critic-agent"),
    before_model_callback=callbacks.log_agent_name_before_callback
)

summarization_agent = Agent(name="summarization_agent",
                            model=config.GEMINI_MODEL,
                            instruction=instructions.get_agent_instructions("summarization-agent"),
                            before_model_callback=callbacks.log_agent_name_before_callback)

root_agent = SequentialAgent(name="root_agent", sub_agents=[research_agent, critic_agent, summarization_agent])

/var/folders/qs/s00ytb510274cnz38ypcfrcr0000gp/T/ipykernel_96223/851088902.py:34: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(name="root_agent", sub_agents=[research_agent, critic_agent, summarization_agent])


# Setup the agent tester

In [26]:
import agent_tester

tester = agent_tester.AgentTester(root_agent)

# Tests

In [27]:
print("================ Top programming languages ===========================")
await tester.run_prompt("Find the top 3 programming languages and describe them.")

================ Top programming languages ===========================
Calling agent research_agent
Calling agent research_agent_tool
Calling agent research_agent
Based on various industry reports and developer surveys, Python, JavaScript, and
Java consistently rank among the top programming languages.                     

Here's a description of each:                                                   

 • Python Python is a dynamic, object-oriented programming language known for   
   combining data structures with easy-to-learn syntax. It is highly valued for 
   its flexibility and is extensively used in fields such as engineering,       
   machine learning, finance, and data science and analysis. Python is          
   frequently chosen by beginner developers and is the most sought-after        
   programming language by recruiters.                                          
 • JavaScript JavaScript (JS) is a scripting language primarily used to make    
   websites and mobile ap

# Perform tests for searching

In [28]:
print("================ Next golf events ===========================")
await tester.run_prompt("Find the next 3 golf events to occur in the next month.")

================ Next golf events ===========================
Calling agent research_agent
Calling agent research_agent_tool
Calling agent research_agent
Here are three golf events scheduled to occur in the next month:                

 1 An event on the Global Amateur Golf Tour is scheduled from August 11 to      
   August 14, 2026.                                                             
 2 The Boeing Classic, part of the PGA TOUR Champions, will take place from     
   August 13 to August 15, 2026.                                                
 3 The FedEx St. Jude Championship, a PGA Tour event, is scheduled for August   
   13-16, 2026.                                                                 
Calling agent critic_agent
🧾 Critique Report                                                              

Agent Evaluated: research_agent Original Request: Find the next 3 golf events to
occur in the next month. Date of Review: October 26, 2023 (Assuming current date
is aroun